# CogAttention — Distractor Filtering

**Track:** Attention — Selective Attention
**Benchmark:** CogAttention v1.0
**Task:** selective

---

## Methodology

Tests selective attention through distractor filtering — extracting verified signal facts while ignoring unverified noise at varying signal-to-noise ratios. Based on Signal-in-Noise (SiN) paradigms.

### Cognitive Science Grounding

- **Signal-in-Noise (SiN) Filtering** — extracting verified facts from a mix of verified (signal) and unverified (noise) sources
- Difficulty scales by making source labels less obvious and increasing noise density, following power-law degradation (GSM-DC, EMNLP 2025)

### Difficulty Scaling

Easy: explicit source tags, 30% noise | Medium: subtle labels, 50% noise | Hard: minimal cues, 60% noise | Expert: implicit cues only, 70% noise | Frontier: zero explicit cues, pure inference required

### Scoring

One assertion per signal value — checks whether the model extracted each verified fact without including unverified distractors.

All instances are procedurally generated from a seed with programmatic ground truth.
No static datasets. 7 layers of contamination resistance including canary strings,
zero lexical overlap (Semantic NIAH), and seed-based regeneration.

---

`<!-- COGATTENTION-BENCH-CANARY-786B240840BC -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Selective Attention
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_selective(response, gold, kbench):
    for signal in gold["signals"]:
        pattern = rf"(?i){_escape_for_regex(signal)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Should extract signal value '{signal}'"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['selective']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_selective")
def cogattention_selective(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention selective task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_selective(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "selective_easy_000",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] Preliminary estimates suggest costs near $8,841.47.\n[Source B (unverified)] Unconfirmed sources report gains of $9,529.02.\n[Source A (verified)] Confirmed quarterly earnings reached $9,512.96.\n[Source B (unverified)] Preliminary estimates suggest costs near $8,948.23.\n[Source A (verified)] Confirmed quarterly earnings reached $6,474.86.\n[Source A (verified)] Confirmed quarterly earnings reached $3,201.33.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"3,201.33\", \"9,512.96\", \"6,474.86\"], \"distractors\": [\"9,529.02\", \"8,841.47\", \"8,948.23\"]}"
 },
 {
  "task_id": "selective_easy_001",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] Speculation puts the figure at $213.58.\n[Source A (verified)] Confirmed quarterly earnings reached $5,163.45.\n[Source B (unverified)] An unaudited draft mentions liabilities of $3,299.56.\n[Source A (verified)] The verified audit found revenue of $2,217.28.\n[Source B (unverified)] Preliminary estimates suggest costs near $2,125.71.\n[Source A (verified)] The validated balance sheet shows assets of $1,951.70.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"2,217.28\", \"5,163.45\", \"1,951.70\"], \"distractors\": [\"3,299.56\", \"213.58\", \"2,125.71\"]}"
 },
 {
  "task_id": "selective_easy_002",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source A (verified)] The accredited lab measured concentration at 148.5 mg/L.\n[Source B (unverified)] A field test kit indicated approximately 73.7 mg/L.\n[Source B (unverified)] A field test kit indicated approximately 68.0 mg/L.\n[Source A (verified)] Quality-controlled testing showed 114.5 mg/L.\n[Source B (unverified)] An uncertified instrument read 52.0 mg/L.\n[Source A (verified)] Certified analysis confirmed 399.7 mg/L of the compound.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"114.5\", \"399.7\", \"148.5\"], \"distractors\": [\"68.0\", \"73.7\", \"52.0\"]}"
 },
 {
  "task_id": "selective_easy_003",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] A passerby estimated the temperature at 9.2°C.\n[Source A (verified)] Official meteorological data shows 9.8°C at noon.\n[Source B (unverified)] An unverified weather app displayed 3.3°C.\n[Source B (unverified)] An unverified weather app displayed 0.4°C.\n[Source A (verified)] The verified sensor reading was -12.7°C.\n[Source A (verified)] The verified sensor reading was 2.5°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-12.7\", \"2.5\", \"9.8\"], \"distractors\": [\"0.4\", \"3.3\", \"9.2\"]}"
 },
 {
  "task_id": "selective_easy_004",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source A (verified)] Quality-controlled testing showed 473.4 mg/L.\n[Source B (unverified)] An uncertified instrument read 152.6 mg/L.\n[Source A (verified)] Quality-controlled testing showed 326.4 mg/L.\n[Source B (unverified)] Preliminary screening suggested 320.7 mg/L.\n[Source A (verified)] The peer-reviewed measurement yielded 351.1 mg/L.\n[Source B (unverified)] A field test kit indicated approximately 21.2 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"326.4\", \"351.1\", \"473.4\"], \"distractors\": [\"21.2\", \"320.7\", \"152.6\"]}"
 },
 {
  "task_id": "selective_easy_005",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] The expired reagent test showed 233.2 mg/L.\n[Source A (verified)] The peer-reviewed measurement yielded 15.6 mg/L.\n[Source A (verified)] Quality-controlled testing showed 218.1 mg/L.\n[Source B (unverified)] A field test kit indicated approximately 457.4 mg/L.\n[Source B (unverified)] Preliminary screening suggested 99.4 mg/L.\n[Source A (verified)] The peer-reviewed measurement yielded 26.5 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"26.5\", \"218.1\", \"15.6\"], \"distractors\": [\"457.4\", \"233.2\", \"99.4\"]}"
 },
 {
  "task_id": "selective_easy_006",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] An uncalibrated thermometer showed 17.2°C.\n[Source B (unverified)] An uncalibrated thermometer showed 1.3°C.\n[Source B (unverified)] Social media posts claimed it felt like 8.0°C.\n[Source A (verified)] The calibrated station recorded a temperature of 3.3°C.\n[Source A (verified)] The verified sensor reading was 31.1°C.\n[Source A (verified)] The verified sensor reading was 33.6°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"3.3\", \"31.1\", \"33.6\"], \"distractors\": [\"17.2\", \"8.0\", \"1.3\"]}"
 },
 {
  "task_id": "selective_easy_007",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] A passerby estimated the temperature at 33.2°C.\n[Source B (unverified)] An uncalibrated thermometer showed 9.6°C.\n[Source A (verified)] The verified sensor reading was 1.8°C.\n[Source A (verified)] The verified sensor reading was 11.3°C.\n[Source B (unverified)] An unverified weather app displayed -10.3°C.\n[Source A (verified)] The calibrated station recorded a temperature of 31.8°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"11.3\", \"1.8\", \"31.8\"], \"distractors\": [\"-10.3\", \"33.2\", \"9.6\"]}"
 },
 {
  "task_id": "selective_medium_008",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource B suggests that Preliminary estimates suggest costs near $8,247.07.\nThe unverified source claims Unconfirmed sources report gains of $3,915.05.\nAccording to Source A, Confirmed quarterly earnings reached $9,689.27.\nAccording to Source B, Unconfirmed sources report gains of $4,846.58.\nSource A reports that Confirmed quarterly earnings reached $4,643.39.\nAccording to Source B, Unconfirmed sources report gains of $8,614.07.\nSource A reports that The validated balance sheet shows assets of $7,978.79.\nSource A reports that According to the certified report, expenses totaled $4,699.28.\nThe unverified source claims Speculation puts the figure at $3,307.16.\nThe unverified source claims An unaudited draft mentions liabilities of $8,036.15.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"4,643.39\", \"7,978.79\", \"4,699.28\", \"9,689.27\"], \"distractors\": [\"3,307.16\", \"8,036.15\", \"8,247.07\", \"3,915.05\", \"4,846.58\", \"8,614.07\"]}"
 },
 {
  "task_id": "selective_medium_009",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe unverified source claims An uncertified instrument read 375.6 mg/L.\nThe verified source states The peer-reviewed measurement yielded 66.7 mg/L.\nThe verified source states The accredited lab measured concentration at 371.7 mg/L.\nThe unverified source claims A field test kit indicated approximately 31.4 mg/L.\nAccording to Source B, A field test kit indicated approximately 328.1 mg/L.\nAccording to Source A, The accredited lab measured concentration at 158.7 mg/L.\nSource A reports that Certified analysis confirmed 122.2 mg/L of the compound.\nThe unverified source claims A field test kit indicated approximately 449.8 mg/L.\nSource B suggests that An uncertified instrument read 128.2 mg/L.\nSource B suggests that A field test kit indicated approximately 416.6 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"122.2\", \"158.7\", \"66.7\", \"371.7\"], \"distractors\": [\"416.6\", \"328.1\", \"375.6\", \"128.2\", \"31.4\", \"449.8\"]}"
 },
 {
  "task_id": "selective_medium_010",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAccording to Source B, The expired reagent test showed 136.2 mg/L.\nAccording to Source B, Preliminary screening suggested 31.2 mg/L.\nSource B suggests that The expired reagent test showed 108.8 mg/L.\nSource A reports that Quality-controlled testing showed 81.8 mg/L.\nAccording to Source A, Quality-controlled testing showed 380.0 mg/L.\nThe unverified source claims A field test kit indicated approximately 185.5 mg/L.\nThe unverified source claims The expired reagent test showed 89.5 mg/L.\nThe unverified source claims The expired reagent test showed 2.8 mg/L.\nAccording to Source A, The peer-reviewed measurement yielded 83.3 mg/L.\nThe verified source states The accredited lab measured concentration at 378.0 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"378.0\", \"380.0\", \"81.8\", \"83.3\"], \"distractors\": [\"136.2\", \"31.2\", \"185.5\", \"108.8\", \"89.5\", \"2.8\"]}"
 },
 {
  "task_id": "selective_medium_011",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource B suggests that An unaudited draft mentions liabilities of $2,017.29.\nAccording to Source B, Speculation puts the figure at $2,233.80.\nThe unverified source claims Speculation puts the figure at $2,019.67.\nAccording to Source A, According to the certified report, expenses totaled $2,462.95.\nAccording to Source A, The verified audit found revenue of $9,468.04.\nSource A reports that Confirmed quarterly earnings reached $124.03.\nSource B suggests that Preliminary estimates suggest costs near $3,874.68.\nSource A reports that The validated balance sheet shows assets of $6,927.26.\nSource B suggests that Unconfirmed sources report gains of $3,925.74.\nThe unverified source claims Unconfirmed sources report gains of $9,129.98.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"2,462.95\", \"9,468.04\", \"6,927.26\", \"124.03\"], \"distractors\": [\"2,233.80\", \"2,017.29\", \"3,925.74\", \"3,874.68\", \"2,019.67\", \"9,129.98\"]}"
 },
 {
  "task_id": "selective_medium_012",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource A reports that The validated balance sheet shows assets of $2,557.13.\nAccording to Source A, The verified audit found revenue of $3,337.46.\nThe unverified source claims Preliminary estimates suggest costs near $4,760.08.\nAccording to Source B, Speculation puts the figure at $8,839.24.\nSource A reports that Confirmed quarterly earnings reached $3,215.74.\nThe verified source states The validated balance sheet shows assets of $2,462.29.\nThe unverified source claims Unconfirmed sources report gains of $9,173.31.\nSource B suggests that An unaudited draft mentions liabilities of $5,907.31.\nSource B suggests that Preliminary estimates suggest costs near $6,307.03.\nAccording to Source B, An unaudited draft mentions liabilities of $8,624.97.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"3,215.74\", \"2,462.29\", \"3,337.46\", \"2,557.13\"], \"distractors\": [\"9,173.31\", \"4,760.08\", \"5,907.31\", \"8,624.97\", \"6,307.03\", \"8,839.24\"]}"
 },
 {
  "task_id": "selective_medium_013",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource B suggests that A passerby estimated the temperature at 4.8°C.\nAccording to Source B, An uncalibrated thermometer showed 21.8°C.\nSource A reports that Official meteorological data shows -3.7°C at noon.\nAccording to Source B, An unverified weather app displayed 11.7°C.\nSource B suggests that A passerby estimated the temperature at 11.5°C.\nSource A reports that The calibrated station recorded a temperature of 30.3°C.\nThe unverified source claims An uncalibrated thermometer showed 34.0°C.\nSource A reports that The calibrated station recorded a temperature of 12.0°C.\nSource B suggests that Social media posts claimed it felt like 38.4°C.\nThe verified source states According to the certified gauge, pressure-adjusted temperature was 23.2°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"30.3\", \"23.2\", \"12.0\", \"-3.7\"], \"distractors\": [\"21.8\", \"11.5\", \"34.0\", \"4.8\", \"11.7\", \"38.4\"]}"
 },
 {
  "task_id": "selective_medium_014",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe unverified source claims Social media posts claimed it felt like 14.6°C.\nSource B suggests that An unverified weather app displayed -13.0°C.\nThe unverified source claims An unverified weather app displayed 23.4°C.\nAccording to Source B, Social media posts claimed it felt like 19.7°C.\nSource B suggests that Social media posts claimed it felt like 30.1°C.\nAccording to Source A, The calibrated station recorded a temperature of 37.0°C.\nSource B suggests that Social media posts claimed it felt like -9.9°C.\nSource A reports that Official meteorological data shows -2.0°C at noon.\nSource A reports that Official meteorological data shows 24.5°C at noon.\nAccording to Source A, The calibrated station recorded a temperature of 31.9°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-2.0\", \"24.5\", \"37.0\", \"31.9\"], \"distractors\": [\"-9.9\", \"14.6\", \"23.4\", \"19.7\", \"-13.0\", \"30.1\"]}"
 },
 {
  "task_id": "selective_medium_015",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource B suggests that An uncertified instrument read 141.4 mg/L.\nThe unverified source claims Preliminary screening suggested 468.4 mg/L.\nSource B suggests that An uncertified instrument read 96.2 mg/L.\nAccording to Source A, The accredited lab measured concentration at 170.1 mg/L.\nThe unverified source claims An uncertified instrument read 54.5 mg/L.\nThe verified source states Certified analysis confirmed 104.1 mg/L of the compound.\nAccording to Source B, A field test kit indicated approximately 29.5 mg/L.\nSource A reports that Quality-controlled testing showed 402.4 mg/L.\nSource A reports that The peer-reviewed measurement yielded 333.8 mg/L.\nAccording to Source B, A field test kit indicated approximately 254.4 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"333.8\", \"170.1\", \"104.1\", \"402.4\"], \"distractors\": [\"54.5\", \"29.5\", \"141.4\", \"96.2\", \"468.4\", \"254.4\"]}"
 },
 {
  "task_id": "selective_hard_016",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nUnofficial channels report Preliminary estimates suggest costs near $5,092.67.\nPer the verified data, According to the certified report, expenses totaled $9,534.96.\nUnofficial channels report An unaudited draft mentions liabilities of $5,868.32.\nCross-referenced documentation shows The validated balance sheet shows assets of $6,132.62.\nThe audited record indicates The validated balance sheet shows assets of $7,364.67.\nUnofficial channels report Preliminary estimates suggest costs near $3,289.23.\nUnconfirmed data indicates Unconfirmed sources report gains of $9,627.12.\nUnofficial channels report Unconfirmed sources report gains of $2,725.10.\nPer the verified data, According to the certified report, expenses totaled $2,779.71.\nUnofficial channels report An unaudited draft mentions liabilities of $5,545.97.\nUnconfirmed data indicates Unconfirmed sources report gains of $873.17.\nAnecdotal evidence suggests An unaudited draft mentions liabilities of $338.75.\nThe audited record indicates The verified audit found revenue of $3,575.48.\nAnecdotal evidence suggests Speculation puts the figure at $6,444.98.\nUnconfirmed data indicates Preliminary estimates suggest costs near $3,109.99.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"3,575.48\", \"2,779.71\", \"9,534.96\", \"7,364.67\", \"6,132.62\"], \"distractors\": [\"338.75\", \"5,545.97\", \"3,109.99\", \"2,725.10\", \"873.17\", \"5,868.32\", \"5,092.67\", \"6,444.98\", \"3,289.23\", \"9,627.12\"]}"
 },
 {
  "task_id": "selective_hard_017",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAnecdotal evidence suggests An unverified weather app displayed 38.6°C.\nCross-referenced documentation shows The calibrated station recorded a temperature of 7.1°C.\nUnconfirmed data indicates An unverified weather app displayed 6.7°C.\nAnecdotal evidence suggests An unverified weather app displayed -1.4°C.\nUnconfirmed data indicates An unverified weather app displayed 21.6°C.\nUnofficial channels report An uncalibrated thermometer showed 29.9°C.\nAnecdotal evidence suggests Social media posts claimed it felt like 20.0°C.\nAnecdotal evidence suggests Social media posts claimed it felt like 16.2°C.\nThe audited record indicates The verified sensor reading was 1.3°C.\nAnecdotal evidence suggests Social media posts claimed it felt like -8.8°C.\nPer the verified data, According to the certified gauge, pressure-adjusted temperature was 29.4°C.\nThe audited record indicates The calibrated station recorded a temperature of 4.3°C.\nUnconfirmed data indicates A passerby estimated the temperature at 28.8°C.\nCross-referenced documentation shows Official meteorological data shows -15.0°C at noon.\nUnofficial channels report A passerby estimated the temperature at 33.4°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"29.4\", \"-15.0\", \"4.3\", \"1.3\", \"7.1\"], \"distractors\": [\"33.4\", \"-8.8\", \"-1.4\", \"6.7\", \"16.2\", \"21.6\", \"38.6\", \"29.9\", \"28.8\", \"20.0\"]}"
 },
 {
  "task_id": "selective_hard_018",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nCross-referenced documentation shows The peer-reviewed measurement yielded 333.1 mg/L.\nThe audited record indicates The accredited lab measured concentration at 42.1 mg/L.\nUnofficial channels report The expired reagent test showed 136.7 mg/L.\nUnofficial channels report A field test kit indicated approximately 235.1 mg/L.\nThe audited record indicates Quality-controlled testing showed 440.6 mg/L.\nAnecdotal evidence suggests Preliminary screening suggested 351.0 mg/L.\nAnecdotal evidence suggests An uncertified instrument read 359.1 mg/L.\nUnconfirmed data indicates A field test kit indicated approximately 242.8 mg/L.\nUnofficial channels report Preliminary screening suggested 178.4 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 353.3 mg/L.\nUnofficial channels report The expired reagent test showed 134.1 mg/L.\nAnecdotal evidence suggests The expired reagent test showed 312.8 mg/L.\nUnconfirmed data indicates An uncertified instrument read 24.6 mg/L.\nThe audited record indicates The accredited lab measured concentration at 49.7 mg/L.\nPer the verified data, The accredited lab measured concentration at 5.6 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"440.6\", \"333.1\", \"49.7\", \"42.1\", \"5.6\"], \"distractors\": [\"134.1\", \"312.8\", \"351.0\", \"359.1\", \"353.3\", \"136.7\", \"242.8\", \"178.4\", \"24.6\", \"235.1\"]}"
 },
 {
  "task_id": "selective_hard_019",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe audited record indicates The peer-reviewed measurement yielded 239.2 mg/L.\nPer the verified data, The accredited lab measured concentration at 355.9 mg/L.\nCross-referenced documentation shows Quality-controlled testing showed 53.3 mg/L.\nUnconfirmed data indicates A field test kit indicated approximately 61.7 mg/L.\nThe audited record indicates The accredited lab measured concentration at 239.5 mg/L.\nCross-referenced documentation shows Quality-controlled testing showed 241.9 mg/L.\nUnofficial channels report Preliminary screening suggested 125.4 mg/L.\nUnofficial channels report The expired reagent test showed 64.1 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 332.7 mg/L.\nUnofficial channels report An uncertified instrument read 288.9 mg/L.\nUnofficial channels report An uncertified instrument read 353.8 mg/L.\nUnofficial channels report A field test kit indicated approximately 411.7 mg/L.\nAnecdotal evidence suggests The expired reagent test showed 473.2 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 174.4 mg/L.\nUnofficial channels report The expired reagent test showed 127.5 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"355.9\", \"53.3\", \"239.2\", \"239.5\", \"241.9\"], \"distractors\": [\"411.7\", \"125.4\", \"353.8\", \"332.7\", \"174.4\", \"64.1\", \"288.9\", \"127.5\", \"473.2\", \"61.7\"]}"
 },
 {
  "task_id": "selective_hard_020",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nUnofficial channels report Unconfirmed sources report gains of $4,369.23.\nUnconfirmed data indicates Speculation puts the figure at $7,617.62.\nAnecdotal evidence suggests Unconfirmed sources report gains of $324.22.\nUnconfirmed data indicates Unconfirmed sources report gains of $9,988.19.\nAnecdotal evidence suggests Preliminary estimates suggest costs near $5,954.72.\nUnofficial channels report Preliminary estimates suggest costs near $2,713.77.\nUnofficial channels report Speculation puts the figure at $7,532.54.\nThe audited record indicates The verified audit found revenue of $4,819.02.\nPer the verified data, The verified audit found revenue of $7,913.23.\nUnconfirmed data indicates An unaudited draft mentions liabilities of $7,942.39.\nUnconfirmed data indicates Preliminary estimates suggest costs near $6,397.21.\nCross-referenced documentation shows The validated balance sheet shows assets of $4,543.29.\nPer the verified data, The validated balance sheet shows assets of $974.85.\nPer the verified data, Confirmed quarterly earnings reached $3,303.87.\nAnecdotal evidence suggests Unconfirmed sources report gains of $9,016.10.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"4,819.02\", \"4,543.29\", \"3,303.87\", \"7,913.23\", \"974.85\"], \"distractors\": [\"324.22\", \"7,942.39\", \"9,988.19\", \"4,369.23\", \"6,397.21\", \"7,532.54\", \"2,713.77\", \"7,617.62\", \"9,016.10\", \"5,954.72\"]}"
 },
 {
  "task_id": "selective_hard_021",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nUnconfirmed data indicates An unaudited draft mentions liabilities of $6,708.89.\nUnconfirmed data indicates Unconfirmed sources report gains of $1,318.56.\nUnconfirmed data indicates Speculation puts the figure at $7,151.85.\nUnofficial channels report Unconfirmed sources report gains of $679.02.\nAnecdotal evidence suggests Unconfirmed sources report gains of $9,018.44.\nUnconfirmed data indicates An unaudited draft mentions liabilities of $5,156.55.\nCross-referenced documentation shows The validated balance sheet shows assets of $2,636.72.\nUnofficial channels report Unconfirmed sources report gains of $7,179.46.\nUnofficial channels report An unaudited draft mentions liabilities of $1,415.11.\nPer the verified data, Confirmed quarterly earnings reached $826.26.\nCross-referenced documentation shows The verified audit found revenue of $4,986.81.\nCross-referenced documentation shows According to the certified report, expenses totaled $6,583.25.\nUnconfirmed data indicates An unaudited draft mentions liabilities of $5,835.33.\nAnecdotal evidence suggests Preliminary estimates suggest costs near $635.12.\nCross-referenced documentation shows According to the certified report, expenses totaled $7,061.29.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"6,583.25\", \"2,636.72\", \"826.26\", \"4,986.81\", \"7,061.29\"], \"distractors\": [\"9,018.44\", \"5,156.55\", \"635.12\", \"679.02\", \"1,415.11\", \"1,318.56\", \"7,151.85\", \"6,708.89\", \"5,835.33\", \"7,179.46\"]}"
 },
 {
  "task_id": "selective_hard_022",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAnecdotal evidence suggests Preliminary screening suggested 177.2 mg/L.\nUnconfirmed data indicates Preliminary screening suggested 416.3 mg/L.\nPer the verified data, Certified analysis confirmed 75.0 mg/L of the compound.\nUnofficial channels report An uncertified instrument read 264.0 mg/L.\nUnofficial channels report Preliminary screening suggested 194.8 mg/L.\nUnofficial channels report The expired reagent test showed 27.7 mg/L.\nCross-referenced documentation shows The peer-reviewed measurement yielded 103.1 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 194.5 mg/L.\nCross-referenced documentation shows Certified analysis confirmed 473.1 mg/L of the compound.\nUnconfirmed data indicates Preliminary screening suggested 486.0 mg/L.\nPer the verified data, The peer-reviewed measurement yielded 75.8 mg/L.\nUnconfirmed data indicates Preliminary screening suggested 406.7 mg/L.\nCross-referenced documentation shows Certified analysis confirmed 221.6 mg/L of the compound.\nAnecdotal evidence suggests The expired reagent test showed 228.2 mg/L.\nUnconfirmed data indicates An uncertified instrument read 412.3 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"75.0\", \"75.8\", \"221.6\", \"103.1\", \"473.1\"], \"distractors\": [\"194.5\", \"177.2\", \"194.8\", \"406.7\", \"264.0\", \"486.0\", \"228.2\", \"412.3\", \"416.3\", \"27.7\"]}"
 },
 {
  "task_id": "selective_hard_023",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nPer the verified data, The accredited lab measured concentration at 387.6 mg/L.\nUnconfirmed data indicates The expired reagent test showed 93.6 mg/L.\nUnofficial channels report An uncertified instrument read 94.0 mg/L.\nPer the verified data, Certified analysis confirmed 106.6 mg/L of the compound.\nUnofficial channels report Preliminary screening suggested 81.5 mg/L.\nAnecdotal evidence suggests An uncertified instrument read 198.2 mg/L.\nAnecdotal evidence suggests An uncertified instrument read 195.0 mg/L.\nUnconfirmed data indicates The expired reagent test showed 446.1 mg/L.\nPer the verified data, The accredited lab measured concentration at 350.2 mg/L.\nUnconfirmed data indicates A field test kit indicated approximately 221.7 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 451.8 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 449.0 mg/L.\nPer the verified data, The peer-reviewed measurement yielded 429.8 mg/L.\nUnconfirmed data indicates An uncertified instrument read 110.5 mg/L.\nThe audited record indicates The peer-reviewed measurement yielded 75.0 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"429.8\", \"106.6\", \"387.6\", \"350.2\", \"75.0\"], \"distractors\": [\"446.1\", \"221.7\", \"451.8\", \"195.0\", \"449.0\", \"93.6\", \"110.5\", \"198.2\", \"81.5\", \"94.0\"]}"
 },
 {
  "task_id": "selective_expert_024",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nIt has been suggested that An unverified weather app displayed 12.5°C.\nRecords show The calibrated station recorded a temperature of 5.3°C.\nReportedly, An unverified weather app displayed -11.6°C.\nSome estimates place A passerby estimated the temperature at -3.7°C.\nDocumentation confirms According to the certified gauge, pressure-adjusted temperature was -4.7°C.\nSome estimates place An unverified weather app displayed -10.3°C.\nThe data indicates The verified sensor reading was 21.6°C.\nSome estimates place An uncalibrated thermometer showed -4.3°C.\nThe data indicates The verified sensor reading was 16.1°C.\nIt has been suggested that An unverified weather app displayed 20.6°C.\nReportedly, A passerby estimated the temperature at 26.0°C.\nSome estimates place A passerby estimated the temperature at -0.6°C.\nSome estimates place An unverified weather app displayed 22.3°C.\nIt has been suggested that A passerby estimated the temperature at -11.7°C.\nReportedly, An uncalibrated thermometer showed 34.9°C.\nDocumentation confirms Official meteorological data shows -14.9°C at noon.\nThe data indicates The calibrated station recorded a temperature of 13.9°C.\nSome estimates place A passerby estimated the temperature at -8.6°C.\nReportedly, Social media posts claimed it felt like 39.4°C.\nIt has been suggested that A passerby estimated the temperature at 33.5°C.\nReportedly, An uncalibrated thermometer showed 39.2°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"21.6\", \"13.9\", \"16.1\", \"-14.9\", \"5.3\", \"-4.7\"], \"distractors\": [\"22.3\", \"34.9\", \"33.5\", \"-0.6\", \"12.5\", \"-8.6\", \"20.6\", \"-11.6\", \"-10.3\", \"39.2\", \"-3.7\", \"-4.3\", \"39.4\", \"26.0\", \"-11.7\"]}"
 },
 {
  "task_id": "selective_expert_025",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nReportedly, Unconfirmed sources report gains of $5,350.77.\nSome estimates place Preliminary estimates suggest costs near $5,839.22.\nIt has been suggested that Unconfirmed sources report gains of $5,682.30.\nRecords show According to the certified report, expenses totaled $7,291.17.\nSome estimates place Preliminary estimates suggest costs near $144.68.\nReportedly, An unaudited draft mentions liabilities of $970.36.\nSome estimates place Unconfirmed sources report gains of $1,725.22.\nReportedly, Speculation puts the figure at $6,899.39.\nReportedly, Preliminary estimates suggest costs near $2,243.98.\nThe data indicates The validated balance sheet shows assets of $8,972.86.\nSome estimates place Unconfirmed sources report gains of $6,398.08.\nReportedly, Unconfirmed sources report gains of $1,529.14.\nRecords show According to the certified report, expenses totaled $5,411.24.\nRecords show The verified audit found revenue of $2,423.93.\nReportedly, An unaudited draft mentions liabilities of $1,825.54.\nRecords show The verified audit found revenue of $8,031.12.\nReportedly, An unaudited draft mentions liabilities of $3,081.84.\nSome estimates place An unaudited draft mentions liabilities of $3,155.84.\nRecords show The validated balance sheet shows assets of $2,113.92.\nReportedly, An unaudited draft mentions liabilities of $6,171.90.\nIt has been suggested that Unconfirmed sources report gains of $4,966.00.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"2,113.92\", \"7,291.17\", \"5,411.24\", \"8,031.12\", \"8,972.86\", \"2,423.93\"], \"distractors\": [\"3,081.84\", \"6,171.90\", \"1,725.22\", \"2,243.98\", \"5,839.22\", \"4,966.00\", \"5,350.77\", \"5,682.30\", \"1,825.54\", \"970.36\", \"3,155.84\", \"1,529.14\", \"6,398.08\", \"6,899.39\", \"144.68\"]}"
 },
 {
  "task_id": "selective_expert_026",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSome estimates place Preliminary screening suggested 235.7 mg/L.\nReportedly, Preliminary screening suggested 116.8 mg/L.\nIt has been suggested that Preliminary screening suggested 62.1 mg/L.\nIt has been suggested that An uncertified instrument read 102.4 mg/L.\nThe data indicates Quality-controlled testing showed 327.3 mg/L.\nIt has been suggested that A field test kit indicated approximately 240.6 mg/L.\nReportedly, A field test kit indicated approximately 397.0 mg/L.\nReportedly, An uncertified instrument read 362.3 mg/L.\nSome estimates place An uncertified instrument read 311.9 mg/L.\nIt has been suggested that A field test kit indicated approximately 475.9 mg/L.\nThe data indicates Certified analysis confirmed 237.6 mg/L of the compound.\nReportedly, A field test kit indicated approximately 435.5 mg/L.\nReportedly, The expired reagent test showed 459.3 mg/L.\nIt has been suggested that Preliminary screening suggested 319.0 mg/L.\nIt has been suggested that An uncertified instrument read 129.6 mg/L.\nDocumentation confirms The accredited lab measured concentration at 73.0 mg/L.\nDocumentation confirms Certified analysis confirmed 470.6 mg/L of the compound.\nIt has been suggested that A field test kit indicated approximately 94.2 mg/L.\nDocumentation confirms Certified analysis confirmed 210.8 mg/L of the compound.\nReportedly, A field test kit indicated approximately 4.7 mg/L.\nDocumentation confirms Quality-controlled testing showed 407.9 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"73.0\", \"327.3\", \"470.6\", \"407.9\", \"210.8\", \"237.6\"], \"distractors\": [\"62.1\", \"319.0\", \"235.7\", \"475.9\", \"129.6\", \"4.7\", \"435.5\", \"362.3\", \"240.6\", \"459.3\", \"397.0\", \"116.8\", \"94.2\", \"311.9\", \"102.4\"]}"
 },
 {
  "task_id": "selective_expert_027",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nRecords show According to the certified gauge, pressure-adjusted temperature was 29.8°C.\nSome estimates place A passerby estimated the temperature at 21.5°C.\nRecords show According to the certified gauge, pressure-adjusted temperature was -6.5°C.\nDocumentation confirms The verified sensor reading was 23.6°C.\nSome estimates place An unverified weather app displayed 2.4°C.\nIt has been suggested that An uncalibrated thermometer showed 34.0°C.\nReportedly, An unverified weather app displayed -8.9°C.\nIt has been suggested that An uncalibrated thermometer showed 1.7°C.\nIt has been suggested that A passerby estimated the temperature at 35.2°C.\nIt has been suggested that A passerby estimated the temperature at 33.9°C.\nThe data indicates Official meteorological data shows -6.1°C at noon.\nReportedly, An uncalibrated thermometer showed 12.6°C.\nSome estimates place Social media posts claimed it felt like 24.7°C.\nThe data indicates According to the certified gauge, pressure-adjusted temperature was -8.1°C.\nIt has been suggested that An unverified weather app displayed 36.6°C.\nIt has been suggested that An unverified weather app displayed 27.4°C.\nReportedly, A passerby estimated the temperature at 18.0°C.\nSome estimates place An unverified weather app displayed 19.6°C.\nSome estimates place An unverified weather app displayed 33.8°C.\nSome estimates place Social media posts claimed it felt like 38.1°C.\nThe data indicates Official meteorological data shows 18.7°C at noon.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"29.8\", \"18.7\", \"-8.1\", \"-6.5\", \"-6.1\", \"23.6\"], \"distractors\": [\"38.1\", \"2.4\", \"19.6\", \"27.4\", \"21.5\", \"35.2\", \"12.6\", \"36.6\", \"-8.9\", \"18.0\", \"34.0\", \"33.9\", \"1.7\", \"33.8\", \"24.7\"]}"
 },
 {
  "task_id": "selective_expert_028",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nIt has been suggested that An uncertified instrument read 126.3 mg/L.\nSome estimates place The expired reagent test showed 215.4 mg/L.\nRecords show The peer-reviewed measurement yielded 21.2 mg/L.\nSome estimates place A field test kit indicated approximately 237.6 mg/L.\nDocumentation confirms The peer-reviewed measurement yielded 231.8 mg/L.\nReportedly, Preliminary screening suggested 137.6 mg/L.\nReportedly, A field test kit indicated approximately 114.3 mg/L.\nThe data indicates Quality-controlled testing showed 126.9 mg/L.\nIt has been suggested that An uncertified instrument read 398.0 mg/L.\nRecords show Quality-controlled testing showed 345.4 mg/L.\nIt has been suggested that A field test kit indicated approximately 434.2 mg/L.\nThe data indicates The peer-reviewed measurement yielded 452.0 mg/L.\nReportedly, A field test kit indicated approximately 13.3 mg/L.\nReportedly, An uncertified instrument read 459.0 mg/L.\nReportedly, A field test kit indicated approximately 320.9 mg/L.\nReportedly, Preliminary screening suggested 385.8 mg/L.\nDocumentation confirms Certified analysis confirmed 355.2 mg/L of the compound.\nSome estimates place A field test kit indicated approximately 332.2 mg/L.\nIt has been suggested that Preliminary screening suggested 49.9 mg/L.\nReportedly, An uncertified instrument read 412.2 mg/L.\nIt has been suggested that An uncertified instrument read 63.7 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"231.8\", \"126.9\", \"355.2\", \"345.4\", \"452.0\", \"21.2\"], \"distractors\": [\"459.0\", \"412.2\", \"114.3\", \"13.3\", \"63.7\", \"385.8\", \"434.2\", \"237.6\", \"137.6\", \"332.2\", \"398.0\", \"49.9\", \"320.9\", \"126.3\", \"215.4\"]}"
 },
 {
  "task_id": "selective_expert_029",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nReportedly, An uncalibrated thermometer showed 21.6°C.\nRecords show According to the certified gauge, pressure-adjusted temperature was -2.4°C.\nIt has been suggested that An unverified weather app displayed -14.9°C.\nIt has been suggested that An unverified weather app displayed -4.6°C.\nReportedly, An uncalibrated thermometer showed 29.6°C.\nReportedly, Social media posts claimed it felt like -0.1°C.\nRecords show According to the certified gauge, pressure-adjusted temperature was 38.0°C.\nIt has been suggested that An unverified weather app displayed 22.9°C.\nReportedly, Social media posts claimed it felt like -2.6°C.\nIt has been suggested that An unverified weather app displayed -10.2°C.\nIt has been suggested that A passerby estimated the temperature at 21.0°C.\nRecords show Official meteorological data shows 6.0°C at noon.\nSome estimates place An unverified weather app displayed -7.3°C.\nThe data indicates The verified sensor reading was 41.3°C.\nIt has been suggested that An unverified weather app displayed 31.4°C.\nDocumentation confirms The calibrated station recorded a temperature of 30.0°C.\nThe data indicates The verified sensor reading was 35.2°C.\nIt has been suggested that A passerby estimated the temperature at 12.1°C.\nSome estimates place A passerby estimated the temperature at 26.8°C.\nReportedly, A passerby estimated the temperature at 34.2°C.\nReportedly, An unverified weather app displayed 33.6°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"38.0\", \"30.0\", \"35.2\", \"41.3\", \"6.0\", \"-2.4\"], \"distractors\": [\"33.6\", \"12.1\", \"26.8\", \"22.9\", \"-10.2\", \"29.6\", \"-4.6\", \"-0.1\", \"-2.6\", \"-14.9\", \"-7.3\", \"21.6\", \"34.2\", \"31.4\", \"21.0\"]}"
 },
 {
  "task_id": "selective_expert_030",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe data indicates Certified analysis confirmed 394.1 mg/L of the compound.\nReportedly, Preliminary screening suggested 162.8 mg/L.\nRecords show Quality-controlled testing showed 457.7 mg/L.\nReportedly, The expired reagent test showed 358.5 mg/L.\nIt has been suggested that A field test kit indicated approximately 265.5 mg/L.\nSome estimates place The expired reagent test showed 446.2 mg/L.\nRecords show The peer-reviewed measurement yielded 489.2 mg/L.\nThe data indicates The peer-reviewed measurement yielded 482.9 mg/L.\nIt has been suggested that A field test kit indicated approximately 105.9 mg/L.\nDocumentation confirms The accredited lab measured concentration at 380.5 mg/L.\nReportedly, The expired reagent test showed 390.4 mg/L.\nReportedly, Preliminary screening suggested 352.3 mg/L.\nIt has been suggested that Preliminary screening suggested 217.5 mg/L.\nSome estimates place A field test kit indicated approximately 439.1 mg/L.\nIt has been suggested that An uncertified instrument read 168.9 mg/L.\nReportedly, An uncertified instrument read 464.2 mg/L.\nIt has been suggested that A field test kit indicated approximately 498.9 mg/L.\nDocumentation confirms Quality-controlled testing showed 7.6 mg/L.\nReportedly, Preliminary screening suggested 165.3 mg/L.\nSome estimates place An uncertified instrument read 327.5 mg/L.\nReportedly, Preliminary screening suggested 115.0 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"489.2\", \"482.9\", \"394.1\", \"457.7\", \"380.5\", \"7.6\"], \"distractors\": [\"358.5\", \"446.2\", \"165.3\", \"327.5\", \"352.3\", \"265.5\", \"498.9\", \"168.9\", \"439.1\", \"105.9\", \"464.2\", \"390.4\", \"162.8\", \"115.0\", \"217.5\"]}"
 },
 {
  "task_id": "selective_expert_031",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nIt has been suggested that An uncalibrated thermometer showed 0.5°C.\nThe data indicates The calibrated station recorded a temperature of 11.9°C.\nDocumentation confirms The calibrated station recorded a temperature of 20.4°C.\nIt has been suggested that Social media posts claimed it felt like 24.0°C.\nThe data indicates Official meteorological data shows 41.1°C at noon.\nReportedly, A passerby estimated the temperature at 18.5°C.\nReportedly, A passerby estimated the temperature at 35.5°C.\nReportedly, An uncalibrated thermometer showed 8.4°C.\nIt has been suggested that A passerby estimated the temperature at 17.3°C.\nSome estimates place A passerby estimated the temperature at 35.1°C.\nDocumentation confirms The calibrated station recorded a temperature of -4.1°C.\nIt has been suggested that A passerby estimated the temperature at -10.2°C.\nDocumentation confirms According to the certified gauge, pressure-adjusted temperature was 3.2°C.\nReportedly, An uncalibrated thermometer showed 23.6°C.\nThe data indicates Official meteorological data shows 10.6°C at noon.\nSome estimates place An uncalibrated thermometer showed 29.5°C.\nReportedly, An unverified weather app displayed 6.3°C.\nSome estimates place A passerby estimated the temperature at 3.7°C.\nReportedly, A passerby estimated the temperature at 7.9°C.\nIt has been suggested that Social media posts claimed it felt like 24.9°C.\nSome estimates place An uncalibrated thermometer showed 15.2°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-4.1\", \"41.1\", \"11.9\", \"20.4\", \"3.2\", \"10.6\"], \"distractors\": [\"35.5\", \"3.7\", \"7.9\", \"23.6\", \"8.4\", \"18.5\", \"29.5\", \"0.5\", \"15.2\", \"24.9\", \"6.3\", \"24.0\", \"-10.2\", \"17.3\", \"35.1\"]}"
 },
 {
  "task_id": "selective_frontier_032",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe findings indicate Unconfirmed sources report gains of $2,146.11.\nThe report states The verified audit found revenue of $2,851.75.\nAccording to the data, An unaudited draft mentions liabilities of $2,219.89.\nAccording to the data, Preliminary estimates suggest costs near $8,232.45.\nThe findings indicate Preliminary estimates suggest costs near $4,418.28.\nAnalysis shows Speculation puts the figure at $4,338.90.\nAccording to the data, An unaudited draft mentions liabilities of $149.99.\nAccording to the data, An unaudited draft mentions liabilities of $5,360.32.\nThe findings indicate The verified audit found revenue of $9,214.08.\nAnalysis shows Preliminary estimates suggest costs near $2,240.47.\nAnalysis shows Unconfirmed sources report gains of $2,441.75.\nThe findings indicate The validated balance sheet shows assets of $8,040.32.\nAnalysis shows Preliminary estimates suggest costs near $118.65.\nThe report states Unconfirmed sources report gains of $977.89.\nAccording to the data, The validated balance sheet shows assets of $8,607.74.\nThe findings indicate An unaudited draft mentions liabilities of $5,390.65.\nAnalysis shows Preliminary estimates suggest costs near $5,613.04.\nAccording to the data, Speculation puts the figure at $5,439.02.\nThe report states Unconfirmed sources report gains of $5,467.27.\nAccording to the data, Unconfirmed sources report gains of $3,634.59.\nAccording to the data, Unconfirmed sources report gains of $3,122.88.\nThe report states Speculation puts the figure at $7,406.96.\nAccording to the data, Confirmed quarterly earnings reached $3,893.00.\nThe report states Speculation puts the figure at $3,864.53.\nThe report states Speculation puts the figure at $4,092.61.\nAnalysis shows The validated balance sheet shows assets of $4,071.84.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"8,040.32\", \"4,071.84\", \"2,851.75\", \"9,214.08\", \"3,893.00\", \"8,607.74\"], \"distractors\": [\"118.65\", \"4,418.28\", \"149.99\", \"3,122.88\", \"5,467.27\", \"977.89\", \"5,613.04\", \"5,390.65\", \"2,146.11\", \"3,864.53\", \"5,439.02\", \"2,219.89\", \"4,338.90\", \"7,406.96\", \"2,240.47\", \"3,634.59\", \"8,232.45\", \"4,092.61\", \"5,360.32\", \"2,441.75\"]}"
 },
 {
  "task_id": "selective_frontier_033",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe findings indicate The verified sensor reading was 13.5°C.\nThe report states Social media posts claimed it felt like 11.1°C.\nThe report states A passerby estimated the temperature at 1.9°C.\nAccording to the data, The calibrated station recorded a temperature of 26.9°C.\nThe findings indicate A passerby estimated the temperature at 27.4°C.\nAnalysis shows Social media posts claimed it felt like -4.4°C.\nThe report states Social media posts claimed it felt like -4.9°C.\nAnalysis shows A passerby estimated the temperature at 25.0°C.\nThe report states Official meteorological data shows 24.1°C at noon.\nAnalysis shows An unverified weather app displayed 3.0°C.\nAccording to the data, Social media posts claimed it felt like -14.7°C.\nThe report states A passerby estimated the temperature at 40.4°C.\nThe findings indicate A passerby estimated the temperature at 16.2°C.\nThe report states Social media posts claimed it felt like -6.6°C.\nAnalysis shows An unverified weather app displayed 31.3°C.\nAccording to the data, A passerby estimated the temperature at 36.3°C.\nThe findings indicate An unverified weather app displayed 27.7°C.\nAnalysis shows Official meteorological data shows -9.9°C at noon.\nThe findings indicate An unverified weather app displayed 5.0°C.\nAccording to the data, An uncalibrated thermometer showed -8.5°C.\nThe report states An uncalibrated thermometer showed 25.6°C.\nThe findings indicate Official meteorological data shows 21.2°C at noon.\nThe findings indicate According to the certified gauge, pressure-adjusted temperature was 8.2°C.\nThe findings indicate An uncalibrated thermometer showed 8.4°C.\nThe findings indicate An unverified weather app displayed 2.5°C.\nThe findings indicate A passerby estimated the temperature at 17.6°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"13.5\", \"21.2\", \"8.2\", \"24.1\", \"-9.9\", \"26.9\"], \"distractors\": [\"-4.4\", \"25.6\", \"8.4\", \"11.1\", \"27.7\", \"16.2\", \"3.0\", \"31.3\", \"2.5\", \"36.3\", \"40.4\", \"25.0\", \"5.0\", \"27.4\", \"-8.5\", \"-4.9\", \"17.6\", \"-6.6\", \"-14.7\", \"1.9\"]}"
 },
 {
  "task_id": "selective_frontier_034",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAccording to the data, An unverified weather app displayed 27.0°C.\nAccording to the data, A passerby estimated the temperature at -1.6°C.\nAnalysis shows A passerby estimated the temperature at 40.1°C.\nThe findings indicate Social media posts claimed it felt like -0.1°C.\nThe findings indicate According to the certified gauge, pressure-adjusted temperature was 20.5°C.\nAnalysis shows An uncalibrated thermometer showed 38.3°C.\nThe findings indicate A passerby estimated the temperature at 21.8°C.\nAccording to the data, The calibrated station recorded a temperature of 19.6°C.\nThe findings indicate An unverified weather app displayed 38.2°C.\nAnalysis shows A passerby estimated the temperature at 35.0°C.\nThe report states The calibrated station recorded a temperature of -9.2°C.\nThe report states An uncalibrated thermometer showed 17.5°C.\nAnalysis shows A passerby estimated the temperature at 30.9°C.\nAnalysis shows Social media posts claimed it felt like 35.5°C.\nThe findings indicate An unverified weather app displayed 15.4°C.\nAccording to the data, Social media posts claimed it felt like -5.6°C.\nThe report states A passerby estimated the temperature at 31.8°C.\nAccording to the data, An uncalibrated thermometer showed -2.4°C.\nThe report states A passerby estimated the temperature at 25.4°C.\nThe findings indicate According to the certified gauge, pressure-adjusted temperature was 28.3°C.\nAnalysis shows Social media posts claimed it felt like 39.0°C.\nAnalysis shows The calibrated station recorded a temperature of 22.7°C.\nThe report states A passerby estimated the temperature at 33.4°C.\nThe report states A passerby estimated the temperature at 32.0°C.\nThe report states According to the certified gauge, pressure-adjusted temperature was 34.6°C.\nAnalysis shows An uncalibrated thermometer showed 33.5°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-9.2\", \"34.6\", \"22.7\", \"19.6\", \"28.3\", \"20.5\"], \"distractors\": [\"39.0\", \"-0.1\", \"31.8\", \"27.0\", \"-1.6\", \"33.4\", \"32.0\", \"25.4\", \"30.9\", \"40.1\", \"38.2\", \"33.5\", \"15.4\", \"21.8\", \"-5.6\", \"35.5\", \"17.5\", \"-2.4\", \"35.0\", \"38.3\"]}"
 },
 {
  "task_id": "selective_frontier_035",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe report states Unconfirmed sources report gains of $9,936.48.\nAnalysis shows According to the certified report, expenses totaled $8,565.59.\nAnalysis shows Unconfirmed sources report gains of $6,535.62.\nAccording to the data, Unconfirmed sources report gains of $9,224.84.\nAccording to the data, An unaudited draft mentions liabilities of $1,821.11.\nThe findings indicate Preliminary estimates suggest costs near $6,452.40.\nThe report states Unconfirmed sources report gains of $4,224.73.\nThe findings indicate Preliminary estimates suggest costs near $7,072.66.\nAnalysis shows Speculation puts the figure at $1,384.81.\nThe report states According to the certified report, expenses totaled $4,405.88.\nAnalysis shows Preliminary estimates suggest costs near $1,398.79.\nAccording to the data, Preliminary estimates suggest costs near $9,546.47.\nThe findings indicate Preliminary estimates suggest costs near $9,253.59.\nThe findings indicate Preliminary estimates suggest costs near $5,499.74.\nAnalysis shows According to the certified report, expenses totaled $3,211.04.\nThe findings indicate Preliminary estimates suggest costs near $8,027.17.\nAnalysis shows Preliminary estimates suggest costs near $8,501.35.\nThe findings indicate Preliminary estimates suggest costs near $4,036.71.\nAnalysis shows Unconfirmed sources report gains of $808.40.\nAnalysis shows An unaudited draft mentions liabilities of $647.52.\nThe findings indicate Unconfirmed sources report gains of $5,662.37.\nAccording to the data, The verified audit found revenue of $4,578.24.\nThe report states The validated balance sheet shows assets of $901.04.\nAccording to the data, Speculation puts the figure at $830.01.\nThe findings indicate An unaudited draft mentions liabilities of $3,087.52.\nThe findings indicate According to the certified report, expenses totaled $3,950.02.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"8,565.59\", \"3,950.02\", \"3,211.04\", \"4,578.24\", \"901.04\", \"4,405.88\"], \"distractors\": [\"6,535.62\", \"8,027.17\", \"1,384.81\", \"7,072.66\", \"647.52\", \"9,546.47\", \"4,224.73\", \"9,936.48\", \"1,398.79\", \"8,501.35\", \"830.01\", \"9,253.59\", \"5,662.37\", \"4,036.71\", \"3,087.52\", \"9,224.84\", \"1,821.11\", \"5,499.74\", \"6,452.40\", \"808.40\"]}"
 },
 {
  "task_id": "selective_frontier_036",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe report states Preliminary screening suggested 467.8 mg/L.\nThe report states Preliminary screening suggested 92.0 mg/L.\nAccording to the data, A field test kit indicated approximately 317.4 mg/L.\nThe report states An uncertified instrument read 3.0 mg/L.\nThe report states Quality-controlled testing showed 432.2 mg/L.\nAnalysis shows Preliminary screening suggested 193.8 mg/L.\nAccording to the data, A field test kit indicated approximately 150.3 mg/L.\nThe report states Preliminary screening suggested 130.8 mg/L.\nAccording to the data, An uncertified instrument read 93.1 mg/L.\nThe findings indicate The peer-reviewed measurement yielded 455.1 mg/L.\nThe report states Certified analysis confirmed 380.9 mg/L of the compound.\nThe findings indicate Quality-controlled testing showed 337.5 mg/L.\nThe report states The expired reagent test showed 111.4 mg/L.\nAccording to the data, An uncertified instrument read 180.4 mg/L.\nThe report states An uncertified instrument read 362.3 mg/L.\nAnalysis shows An uncertified instrument read 222.8 mg/L.\nAnalysis shows An uncertified instrument read 274.8 mg/L.\nThe findings indicate Preliminary screening suggested 299.0 mg/L.\nAnalysis shows The accredited lab measured concentration at 306.7 mg/L.\nThe report states The expired reagent test showed 36.9 mg/L.\nThe findings indicate The expired reagent test showed 241.1 mg/L.\nThe report states A field test kit indicated approximately 413.7 mg/L.\nThe report states The expired reagent test showed 82.8 mg/L.\nThe report states Certified analysis confirmed 408.1 mg/L of the compound.\nAnalysis shows An uncertified instrument read 301.1 mg/L.\nAnalysis shows Preliminary screening suggested 245.7 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"432.2\", \"337.5\", \"306.7\", \"408.1\", \"380.9\", \"455.1\"], \"distractors\": [\"362.3\", \"180.4\", \"3.0\", \"301.1\", \"93.1\", \"193.8\", \"130.8\", \"82.8\", \"111.4\", \"467.8\", \"222.8\", \"241.1\", \"150.3\", \"36.9\", \"317.4\", \"413.7\", \"299.0\", \"274.8\", \"245.7\", \"92.0\"]}"
 },
 {
  "task_id": "selective_frontier_037",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe report states The accredited lab measured concentration at 359.3 mg/L.\nAccording to the data, Preliminary screening suggested 268.8 mg/L.\nThe findings indicate A field test kit indicated approximately 452.3 mg/L.\nThe findings indicate Preliminary screening suggested 488.5 mg/L.\nThe report states Preliminary screening suggested 75.2 mg/L.\nAnalysis shows Quality-controlled testing showed 198.2 mg/L.\nThe report states An uncertified instrument read 61.2 mg/L.\nAnalysis shows An uncertified instrument read 416.3 mg/L.\nThe findings indicate Certified analysis confirmed 270.9 mg/L of the compound.\nAccording to the data, A field test kit indicated approximately 191.4 mg/L.\nThe findings indicate The accredited lab measured concentration at 220.4 mg/L.\nThe findings indicate Preliminary screening suggested 230.8 mg/L.\nAccording to the data, The peer-reviewed measurement yielded 123.0 mg/L.\nAnalysis shows A field test kit indicated approximately 261.0 mg/L.\nThe report states Preliminary screening suggested 10.0 mg/L.\nAccording to the data, Preliminary screening suggested 353.5 mg/L.\nThe report states The peer-reviewed measurement yielded 275.0 mg/L.\nThe findings indicate The expired reagent test showed 17.6 mg/L.\nAccording to the data, Preliminary screening suggested 257.7 mg/L.\nThe report states Preliminary screening suggested 416.9 mg/L.\nAccording to the data, The expired reagent test showed 89.2 mg/L.\nAnalysis shows Preliminary screening suggested 478.6 mg/L.\nThe findings indicate A field test kit indicated approximately 390.8 mg/L.\nThe findings indicate An uncertified instrument read 245.3 mg/L.\nAnalysis shows Preliminary screening suggested 96.3 mg/L.\nThe findings indicate The expired reagent test showed 231.8 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"198.2\", \"123.0\", \"275.0\", \"359.3\", \"220.4\", \"270.9\"], \"distractors\": [\"261.0\", \"231.8\", \"96.3\", \"61.2\", \"268.8\", \"416.3\", \"75.2\", \"230.8\", \"17.6\", \"10.0\", \"245.3\", \"478.6\", \"416.9\", \"89.2\", \"452.3\", \"257.7\", \"353.5\", \"191.4\", \"390.8\", \"488.5\"]}"
 },
 {
  "task_id": "selective_frontier_038",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAnalysis shows A passerby estimated the temperature at 37.0°C.\nThe findings indicate The calibrated station recorded a temperature of 24.1°C.\nAnalysis shows A passerby estimated the temperature at -2.6°C.\nAccording to the data, A passerby estimated the temperature at 17.9°C.\nAnalysis shows A passerby estimated the temperature at 5.4°C.\nAccording to the data, An unverified weather app displayed 41.7°C.\nThe report states Social media posts claimed it felt like -2.2°C.\nThe findings indicate The verified sensor reading was -9.3°C.\nAccording to the data, A passerby estimated the temperature at 4.8°C.\nAnalysis shows Social media posts claimed it felt like 12.1°C.\nAccording to the data, According to the certified gauge, pressure-adjusted temperature was 24.9°C.\nThe report states A passerby estimated the temperature at 7.9°C.\nAccording to the data, An uncalibrated thermometer showed 16.6°C.\nAccording to the data, An uncalibrated thermometer showed 17.3°C.\nAnalysis shows A passerby estimated the temperature at 40.9°C.\nAnalysis shows An unverified weather app displayed 14.0°C.\nAccording to the data, A passerby estimated the temperature at 40.4°C.\nThe report states An uncalibrated thermometer showed 39.9°C.\nAnalysis shows An unverified weather app displayed 1.2°C.\nAccording to the data, An uncalibrated thermometer showed 12.0°C.\nAnalysis shows The verified sensor reading was -2.4°C.\nThe findings indicate Social media posts claimed it felt like -13.3°C.\nThe report states A passerby estimated the temperature at 37.7°C.\nThe findings indicate According to the certified gauge, pressure-adjusted temperature was 25.3°C.\nAnalysis shows The calibrated station recorded a temperature of 9.9°C.\nAccording to the data, An uncalibrated thermometer showed -0.3°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-2.4\", \"24.1\", \"24.9\", \"-9.3\", \"9.9\", \"25.3\"], \"distractors\": [\"16.6\", \"41.7\", \"-2.6\", \"14.0\", \"5.4\", \"-0.3\", \"17.3\", \"-2.2\", \"-13.3\", \"39.9\", \"40.9\", \"1.2\", \"37.0\", \"40.4\", \"7.9\", \"12.1\", \"17.9\", \"4.8\", \"37.7\", \"12.0\"]}"
 },
 {
  "task_id": "selective_frontier_039",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAccording to the data, A passerby estimated the temperature at -13.6°C.\nAccording to the data, An unverified weather app displayed 39.3°C.\nThe report states The calibrated station recorded a temperature of 32.2°C.\nAnalysis shows An unverified weather app displayed 32.1°C.\nAccording to the data, The calibrated station recorded a temperature of -3.0°C.\nThe findings indicate An uncalibrated thermometer showed 27.0°C.\nAccording to the data, Official meteorological data shows 40.8°C at noon.\nAnalysis shows An uncalibrated thermometer showed 5.9°C.\nAccording to the data, An uncalibrated thermometer showed 39.7°C.\nThe findings indicate A passerby estimated the temperature at 16.0°C.\nThe report states An unverified weather app displayed 13.6°C.\nAnalysis shows An uncalibrated thermometer showed 18.9°C.\nThe report states An unverified weather app displayed 23.5°C.\nAccording to the data, A passerby estimated the temperature at 15.8°C.\nAccording to the data, An uncalibrated thermometer showed 12.4°C.\nThe findings indicate A passerby estimated the temperature at -9.5°C.\nThe report states According to the certified gauge, pressure-adjusted temperature was 32.4°C.\nThe report states An uncalibrated thermometer showed -7.9°C.\nAnalysis shows An unverified weather app displayed 17.7°C.\nThe findings indicate A passerby estimated the temperature at 31.4°C.\nAccording to the data, The verified sensor reading was 18.4°C.\nThe report states The calibrated station recorded a temperature of 12.7°C.\nThe report states An uncalibrated thermometer showed 5.7°C.\nThe findings indicate Social media posts claimed it felt like 35.1°C.\nThe report states Social media posts claimed it felt like 10.3°C.\nAnalysis shows An unverified weather app displayed 33.9°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"12.7\", \"-3.0\", \"32.2\", \"40.8\", \"18.4\", \"32.4\"], \"distractors\": [\"15.8\", \"31.4\", \"12.4\", \"-7.9\", \"33.9\", \"-13.6\", \"39.3\", \"39.7\", \"32.1\", \"5.9\", \"18.9\", \"16.0\", \"13.6\", \"17.7\", \"-9.5\", \"10.3\", \"35.1\", \"23.5\", \"5.7\", \"27.0\"]}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['selective']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "selective": cogattention_selective,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Selective Attention")
